# Adversarial Machine Learning: Attacking and Defending a Digit Classifier

This notebook is self-contained. It trains a classifier on MNIST digits, attacks it
with **FGSM** and **PGD**, then defends it with **adversarial training**.

**How to run it on Google Colab**
1. Open the notebook in Colab (File > Upload notebook, or open it from Google Drive).
2. Turn on the GPU: **Runtime > Change runtime type > Hardware accelerator > T4 GPU**.
3. Run everything: **Runtime > Run all**.
4. Wait for the last cell to finish. The full run takes roughly 10 to 20 minutes on a T4 GPU.
5. Plots appear inline. The plot images and trained model files are saved in the `results` and `models` folders, and the last cell packages them into `adversarial_ml_results.zip` and offers it for download.

Nothing needs to be installed or uploaded. The dataset downloads automatically.

## 0. Setup

Imports, the device check, and the settings used throughout. Colab already includes PyTorch, torchvision and matplotlib.

In [ ]:
import os
import random
import shutil

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cpu":
    print("WARNING: no GPU found. Go to Runtime > Change runtime type > T4 GPU, "
          "or the adversarial training step will be very slow.")

SEED = 0
DATA_DIR = "data"
RESULTS_DIR = "results"
MODELS_DIR = "models"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)


def set_seed(seed):
    """Make the random parts (weight init, shuffling, PGD random starts) repeatable."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)

# Attack and training settings
BASELINE_EPOCHS = 3
ADV_EPOCHS = 8
BATCH_SIZE = 128
EVAL_BATCH_SIZE = 256
LEARNING_RATE = 1e-3
EPSILONS = [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3]
FGSM_EPS_FOR_EXAMPLES = 0.2
PGD_EVAL_STEPS = 20
EPS_TRAIN = 0.3        # attack strength the defended model is trained against
RAMP_EPOCHS = 2        # epsilon warms up from 0 to EPS_TRAIN over this many epochs
TRAIN_PGD_STEPS = 10   # fewer PGD steps during training, to keep it fast

## 1. The model

A small convolutional neural network (CNN). Its input is a batch of images with pixel values in **[0, 1]**.
The mean/std normalization happens *inside* the model, so attack sizes (epsilon) are measured on the
raw 0–1 pixel scale.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Normalization is a fixed step, not a learned one.
        self.register_buffer("mean", torch.tensor(0.1307).view(1, 1, 1, 1))
        self.register_buffer("std", torch.tensor(0.3081).view(1, 1, 1, 1))

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),   # 28x28 -> 28x28
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 28x28 -> 14x14
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # 14x14 -> 14x14
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 14x14 -> 7x7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10),  # 10 digits -> 10 raw scores (logits)
        )

    def forward(self, x):
        x = (x - self.mean) / self.std
        return self.classifier(self.features(x))


print("Parameters:", sum(p.numel() for p in SmallCNN().parameters()))

## 2. Data

MNIST: 60,000 training images and 10,000 test images. `ToTensor()` turns each pixel into a float in [0, 1].

In [ ]:
to_tensor = transforms.ToTensor()
train_set = datasets.MNIST(DATA_DIR, train=True, download=True, transform=to_tensor)
test_set = datasets.MNIST(DATA_DIR, train=False, download=True, transform=to_tensor)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_set, batch_size=EVAL_BATCH_SIZE)
print("Training images:", len(train_set), "  Test images:", len(test_set))

## 3. Baseline model (normal training)

Trained the usual way: the gradient is taken with respect to the **weights**, and the weights move to lower the loss.

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    """Clean accuracy: fraction of images classified correctly."""
    model.eval()
    correct, total = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return correct / total


def train_normal(epochs=BASELINE_EPOCHS):
    set_seed(SEED)
    model = SmallCNN().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(images), labels)
            loss.backward()          # gradients with respect to the WEIGHTS
            optimizer.step()
            running_loss += loss.item() * labels.size(0)
        acc = evaluate(model, test_loader)
        print(f"Epoch {epoch}/{epochs}  train loss {running_loss / len(train_set):.4f}  "
              f"test acc {acc:.4f}")
    return model


baseline = train_normal()
torch.save(baseline.state_dict(), os.path.join(MODELS_DIR, "baseline_cnn.pt"))
print(f"Final clean test accuracy of the baseline: {evaluate(baseline, test_loader) * 100:.2f}%")

## 4. FGSM attack (Fast Gradient Sign Method)

One step. Compute the gradient of the loss with respect to each **input pixel**, keep only its sign (+1 or −1),
then move every pixel by epsilon in that direction:

    x_adv = x + epsilon * sign(gradient of loss with respect to x)

The model stays frozen. Only the image changes.

In [ ]:
def fgsm_attack(model, images, labels, epsilon, loss_fn):
    images = images.clone().detach().requires_grad_(True)
    loss = loss_fn(model(images), labels)
    # Gradient with respect to the PIXELS, not the weights.
    grad = torch.autograd.grad(loss, images)[0]
    adv_images = images + epsilon * grad.sign()
    return adv_images.clamp(0, 1).detach()


def accuracy_under_attack(model, loader, attack=None):
    """Accuracy on `loader`. If `attack` is given, each batch is attacked first."""
    model.eval()
    correct, total = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        if attack is not None:
            images = attack(images, labels)
        with torch.no_grad():
            preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return correct / total


loss_fn = nn.CrossEntropyLoss()

fgsm_accs = []
for eps in EPSILONS:
    acc = accuracy_under_attack(
        baseline, test_loader,
        attack=lambda x, y, e=eps: fgsm_attack(baseline, x, y, e, loss_fn))
    fgsm_accs.append(acc)
    print(f"FGSM  epsilon {eps:.2f}  ->  accuracy {acc * 100:.2f}%")

plt.figure(figsize=(7, 4.5))
plt.plot(EPSILONS, [a * 100 for a in fgsm_accs], marker="o")
plt.xlabel("epsilon (max pixel change, on the 0-1 scale)")
plt.ylabel("accuracy (%)")
plt.title("FGSM: model accuracy vs. attack strength")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fgsm_accuracy_vs_epsilon.png"), dpi=150)
plt.show()

In [ ]:
# Example images: clean vs. FGSM-attacked, with predictions. Red = the attack fooled the model.
sample_images, sample_labels = next(iter(test_loader))
n = 8
imgs, lbls = sample_images[:n].to(device), sample_labels[:n].to(device)
adv = fgsm_attack(baseline, imgs, lbls, FGSM_EPS_FOR_EXAMPLES, loss_fn)
with torch.no_grad():
    clean_preds = baseline(imgs).argmax(dim=1)
    adv_preds = baseline(adv).argmax(dim=1)

fig, axes = plt.subplots(2, n, figsize=(n * 1.6, 4.2))
for i in range(n):
    axes[0, i].imshow(imgs[i, 0].cpu(), cmap="gray", vmin=0, vmax=1)
    axes[0, i].set_title(f"clean: {clean_preds[i].item()}", fontsize=9)
    axes[1, i].imshow(adv[i, 0].cpu(), cmap="gray", vmin=0, vmax=1)
    color = "red" if adv_preds[i] != lbls[i] else "green"
    axes[1, i].set_title(f"adv: {adv_preds[i].item()}", fontsize=9, color=color)
    axes[0, i].axis("off")
    axes[1, i].axis("off")
fig.suptitle(f"FGSM examples at epsilon = {FGSM_EPS_FOR_EXAMPLES}  (red = attack fooled the model)")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fgsm_examples.png"), dpi=150)
plt.show()

## 5. PGD attack (Projected Gradient Descent)

FGSM repeated many times with small steps. After each step, every pixel is **projected** back into the
epsilon-ball around the original image, so no pixel changes by more than epsilon. The attack also starts
from a random point inside the ball. This is stronger than FGSM because it re-checks the gradient at every step.

In [ ]:
def pgd_attack(model, images, labels, epsilon, loss_fn, steps=PGD_EVAL_STEPS, random_start=True):
    alpha = 2.5 * epsilon / steps          # step size
    original = images.clone().detach()

    if random_start:
        adv = original + torch.empty_like(original).uniform_(-epsilon, epsilon)
        adv = adv.clamp(0, 1)
    else:
        adv = original.clone()

    for _ in range(steps):
        adv.requires_grad_(True)
        loss = loss_fn(model(adv), labels)
        grad = torch.autograd.grad(loss, adv)[0]
        adv = adv.detach() + alpha * grad.sign()
        # Projection: stay within epsilon of the original image, and within [0, 1].
        adv = torch.max(torch.min(adv, original + epsilon), original - epsilon)
        adv = adv.clamp(0, 1)
    return adv.detach()


set_seed(0)
pgd_accs = []
for eps in EPSILONS:
    acc = accuracy_under_attack(
        baseline, test_loader,
        attack=lambda x, y, e=eps: pgd_attack(baseline, x, y, e, loss_fn))
    pgd_accs.append(acc)
    print(f"epsilon {eps:.2f}  FGSM {fgsm_accs[EPSILONS.index(eps)] * 100:6.2f}%   "
          f"PGD {acc * 100:6.2f}%")

plt.figure(figsize=(7, 4.5))
plt.plot(EPSILONS, [a * 100 for a in fgsm_accs], marker="o", label="FGSM (1 step)")
plt.plot(EPSILONS, [a * 100 for a in pgd_accs], marker="s", label=f"PGD ({PGD_EVAL_STEPS} steps)")
plt.xlabel("epsilon (max pixel change, on the 0-1 scale)")
plt.ylabel("accuracy (%)")
plt.title("Attack strength: FGSM vs. PGD")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fgsm_vs_pgd.png"), dpi=150)
plt.show()

In [ ]:
# Optional cross-check against the torchattacks library. Colab does not include it, so this cell
# skips the check if the library is missing. Nothing needs to be installed for the rest of the notebook.
try:
    import torchattacks
except ImportError:
    torchattacks = None

if torchattacks is None:
    print("torchattacks is not installed, so the library cross-check is skipped. "
          "The attacks above are our own implementations.")
else:
    eps = 0.2
    ours = accuracy_under_attack(
        baseline, test_loader,
        attack=lambda x, y: pgd_attack(baseline, x, y, eps, loss_fn, random_start=False))
    library = torchattacks.PGD(baseline, eps=eps, alpha=2.5 * eps / PGD_EVAL_STEPS,
                               steps=PGD_EVAL_STEPS, random_start=False)
    lib = accuracy_under_attack(baseline, test_loader, attack=lambda x, y: library(x, y))
    print(f"Cross-check at epsilon 0.20 (no random start): ours {ours * 100:.2f}%  "
          f"vs torchattacks {lib * 100:.2f}%")

## 6. Adversarial training (the defense)

Each batch is attacked with PGD against the current model, and the model is trained on the attacked images
with their correct labels, so it learns to classify them.

Epsilon starts at 0 and ramps up to 0.3. Training straight at 0.3 from random weights collapses to guessing
(this was tested in the project: the loss stuck at ln 10 = 2.3013).

In [ ]:
def train_adversarial(epochs=ADV_EPOCHS):
    set_seed(SEED)
    model = SmallCNN().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    total_batches = len(train_loader)

    for epoch in range(1, epochs + 1):
        running_loss = 0.0
        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            # Epsilon warm-up
            progress = ((epoch - 1) + batch_idx / total_batches) / RAMP_EPOCHS
            eps_now = EPS_TRAIN * min(1.0, progress)

            # Step A: attack the current model.
            model.eval()
            adv_images = pgd_attack(model, images, labels, eps_now, loss_fn,
                                    steps=TRAIN_PGD_STEPS)

            # Step B: train on the attacked images.
            model.train()
            optimizer.zero_grad()
            loss = loss_fn(model(adv_images), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * labels.size(0)

        clean_acc = evaluate(model, test_loader)
        print(f"Epoch {epoch}/{epochs}  eps {eps_now:.3f}  "
              f"adv train loss {running_loss / len(train_set):.4f}  clean test acc {clean_acc:.4f}")
    model.eval()
    return model


adv_model = train_adversarial()
torch.save(adv_model.state_dict(), os.path.join(MODELS_DIR, "adv_cnn.pt"))
print("Saved the adversarially trained model.")

## 7. Results: baseline vs. adversarially trained

Both models are attacked with the same PGD-20 attack.

In [ ]:
set_seed(0)
base_accs = [accuracy_under_attack(
                 baseline, test_loader,
                 attack=lambda x, y, e=eps: pgd_attack(baseline, x, y, e, loss_fn))
             for eps in EPSILONS]
set_seed(0)
adv_accs = [accuracy_under_attack(
                adv_model, test_loader,
                attack=lambda x, y, e=eps: pgd_attack(adv_model, x, y, e, loss_fn))
            for eps in EPSILONS]

print(f"{'epsilon':>8}  {'baseline':>10}  {'adversarial':>12}")
for eps, b, a in zip(EPSILONS, base_accs, adv_accs):
    print(f"{eps:8.2f}  {b * 100:9.2f}%  {a * 100:11.2f}%")

plt.figure(figsize=(7, 4.5))
plt.plot(EPSILONS, [a * 100 for a in base_accs], marker="o", label="baseline (normal training)")
plt.plot(EPSILONS, [a * 100 for a in adv_accs], marker="s", label="adversarially trained")
plt.xlabel("epsilon (max pixel change, on the 0-1 scale)")
plt.ylabel(f"accuracy under PGD-{PGD_EVAL_STEPS} (%)")
plt.title("Robustness: baseline vs. adversarial training")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "baseline_vs_adv_robustness.png"), dpi=150)
plt.show()

## 8. Download the results

Packages the plots and model files into one zip. On Colab, the zip also downloads to your computer.
If you are not on Colab, the files are still in the `results` and `models` folders.

In [ ]:
shutil.rmtree("adversarial_ml_results_tmp", ignore_errors=True)
os.makedirs("adversarial_ml_results_tmp")
shutil.copytree(RESULTS_DIR, "adversarial_ml_results_tmp/results")
shutil.copytree(MODELS_DIR, "adversarial_ml_results_tmp/models")
zip_path = shutil.make_archive("adversarial_ml_results", "zip", "adversarial_ml_results_tmp")
shutil.rmtree("adversarial_ml_results_tmp")
print("Created", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Not running on Colab. Find the zip in the notebook's folder.")